# **Corndel | AI6 Level 6 ML Engineer Apprenticeship**
### 7.4 From Metrics to Decisions


Rail vehicles run on axle bearings that develop faults gradually through fatigue, cracking, or pitting. Caught early, a defective bearing is a scheduled replacement. Missed, it can lead to axle failure in service: a safety incident and passengers on a train that should not have run.

Manual inspection at scale is not feasible, so a computer vision classifier flags bearing images for human review. An engineer makes the final call.

The model determines which bearings are flagged for the engineer's inspection, and which are cleared.

**Your team has two candidate classifier models. Both were trained on the same dataset and evaluated on the same 500-image holdout set. One need to be chosen. That recommendation is yours to make.**
<br><br>

---

*Run each cell in order using Shift+Enter. The notebook will tell you when to stop and write.*

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import (
    confusion_matrix, accuracy_score, precision_score,
    recall_score, f1_score, precision_recall_curve,
)
from utils import (
    plot_confusion_matrices,
    plot_pr_curves,
    operational_summary,
    validate_recommendation,
)

## Loading the data

The dataset contains pre-computed evaluation results for 500 bearing inspection images. Of these, 100 contain confirmed defects. Each row records the true label and both models' predicted probability that the image contains a defect.

In [ ]:
df = pd.read_csv("bearings.csv")

print(f"Shape: {df.shape}")
print(f"\nClass distribution:")
print(df["true_label"].value_counts().rename({0: "No defect", 1: "Defect"}))
print(f"\nDefect rate: {df['true_label'].mean():.0%}")

In [ ]:
fig, ax = plt.subplots(figsize=(6, 4))
df["true_label"].value_counts().plot(
    kind="bar", ax=ax, color=["steelblue", "coral"]
)
ax.set_xticklabels(["No defect", "Defect"], rotation=0)
ax.set_ylabel("Count")
ax.set_title("Class distribution in holdout set")
plt.tight_layout()
plt.show()

## Metrics at the default threshold

Both models output a probability between 0 and 1. At the default threshold of 0.5, any image scoring 0.5 or above is flagged as a defect.

In [ ]:
y      = df["true_label"].values
prob_a = df["model_a_prob"].values
prob_b = df["model_b_prob"].values

pred_a = (prob_a >= 0.5).astype(int)
pred_b = (prob_b >= 0.5).astype(int)

metrics = pd.DataFrame({
    "Model A": {
        "Accuracy":  accuracy_score(y, pred_a),
        "Precision": precision_score(y, pred_a),
        "Recall":    recall_score(y, pred_a),
        "F1":        f1_score(y, pred_a),
    },
    "Model B": {
        "Accuracy":  accuracy_score(y, pred_b),
        "Precision": precision_score(y, pred_b),
        "Recall":    recall_score(y, pred_b),
        "F1":        f1_score(y, pred_b),
    },
})

metrics.round(3)

Look at the two columns. Model A has higher precision. Model B has higher
recall. Their accuracy is almost identical.

None of those four numbers tells you which model to deploy. Before you can
answer that question, you need to look at what is hiding behind the summary.

### Confusion matrices

The summary metrics hide the raw counts. Pay attention to the **false negatives** (FN): those are defective bearings the model cleared for service.

In [ ]:
cm_a = confusion_matrix(y, pred_a)
cm_b = confusion_matrix(y, pred_b)

# Pandas table: the raw counts
def cm_table(cm, name):
    tn, fp, fn, tp = cm.ravel()
    return pd.DataFrame(
        [[tn, fp], [fn, tp]],
        index=["Actual: no defect", "Actual: defect"],
        columns=["Pred: no defect", "Pred: defect"],
    )

print("Model A")
display(cm_table(cm_a, "Model A"))
print("\nModel B")
display(cm_table(cm_b, "Model B"))

The table gives you the counts. The visual below makes one pattern harder to miss.

In [ ]:
plot_confusion_matrices(cm_a, cm_b)

## Which errors matter more?

Both models have high accuracy, but they make different kinds of mistakes.

A **false positive** means a healthy bearing gets flagged. An engineer inspects it, finds nothing wrong, and sends it back into service. That costs time and money.

A **false negative** means a defective bearing gets cleared. It goes back into service and continues to degrade. If the fault is serious enough, the bearing fails while the train is running.

These are not equivalent mistakes. The costs fall on different people and with very different consequences.

## The precision-recall trade-off

The metrics above assume a threshold of 0.5, but there is nothing fixed about that number. Lowering the threshold catches more defects (higher recall) at the cost of more false alarms (lower precision). The precision-recall curve shows this trade-off across every possible threshold.

In [ ]:
plot_pr_curves(y, prob_a, prob_b)

Model B's curve sits above and to the right of Model A's for most of the range, meaning it can achieve higher recall at the same precision, or higher precision at the same recall.

The steep drop at the right-hand end is where the last few defects look almost identical to healthy bearings. No amount of threshold tuning will catch them cleanly. If those cases matter, you need a better model, not a lower threshold.

*Click the triangle icon to the left to expand the section below.*

<details>
<summary>Why does this trade-off exist in every classifier ever built?</summary>

It is not a flaw in the model. It is a property of the world.

Defective and healthy bearings do not come with labels attached. A classifier learns to distinguish them by finding patterns in images. But some defective bearings look almost identical to healthy ones: early-stage pitting at low contrast, for instance. Any threshold you set will either catch some of those borderline cases and pull in false alarms alongside them, or miss them and stay precise.

This same tension appears in medical diagnosis, spam filters, fraud detection, and radar systems. A radar set to detect every incoming aircraft will trigger on flocks of birds. One set to ignore birds will miss low-flying aircraft. There is no setting that solves this. There is only a choice about which error you are more willing to make.

The precision-recall curve is just a map of every version of that choice.

</details>

*Click the triangle icon to the left to expand the section below.*

<details>
<summary>Why does the curve look like this? Four questions worth asking.</summary>

Every point on these curves is a possible deployment configuration. Moving along the curve is not free: gaining precision costs recall, and vice versa.

| | |
|:--|:--|
| **Why does the curve have steps?** | The holdout set has 100 defects. Each one caught moves recall by exactly 0.01. A smooth curve needs thousands of positives. Yours does not have them. |
| **Why does Model B hold precision longer?** | Its probability scores are better calibrated for high-recall operation. Model A starts trading precision for recall earlier. That is why its default point sits at 66% recall despite high precision. |
| **What does the cliff tell you?** | The remaining defects look almost identical to clean bearings. No threshold adjustment will catch them cleanly. If those cases matter, the answer is a better model, not a tuned threshold. |
| **Is there a point where you would stop?** | Pushing recall from 90% to 100% costs a large number of false alarms. Whether that is acceptable depends on who bears each type of error, which is exactly what sections 1 to 5 ask you to work out. |

</details>

## Exploring the threshold

Change `THRESHOLD` in the cell below and re-run it to see how the metrics shift for Model B.

**Task:** Find a threshold where Model B roughly matches Model A's default precision (~0.94). What happens to recall?

In [ ]:
# Change this value and re-run
THRESHOLD = 0.50  # try values between 0.30 and 0.90

pred_b_t = (prob_b >= THRESHOLD).astype(int)

print(f"Model B at threshold {THRESHOLD}")
print(f"  Precision : {precision_score(y, pred_b_t):.3f}   (Model A default: {precision_score(y, pred_a):.3f})")
print(f"  Recall    : {recall_score(y, pred_b_t):.3f}   (Model A default: {recall_score(y, pred_a):.3f})")
print()
cm_bt = confusion_matrix(y, pred_b_t)
cm_ad = confusion_matrix(y, pred_a)
print(f"  Missed defects : {cm_bt[1, 0]}   (Model A default: {cm_ad[1, 0]})")
print(f"  False alarms   : {cm_bt[0, 1]}   (Model A default: {cm_ad[0, 1]})")

What did you find? Note your threshold value and what it cost in recall. You will use this when you fill in section 6.

## Putting the numbers in context

Metrics on a holdout set are abstract. The cell below translates them into missed defects per week and per year on a busy route processing 2,000 inspection images. Both models are compared at the default threshold of 0.5 so the baseline is fair.

In [ ]:
operational_summary(y, pred_a, pred_b)

You now have the full technical picture.

Before you make a recommendation, there is one more thing to consider.

A few things that may help with sections 1 to 5:

- The formal recourse routes for a harmed passenger are the Rail Accident Investigation Branch (RAIB), the Office of Rail and Road (ORR), and civil litigation. Consider honestly whether these are realistic for an ordinary person without legal support.
- On accountability there is no single correct answer. You may want to search *who is responsible for algorithmic deployment decisions* and there is genuine ongoing debate.

Open `recommendation.md` in the file browser on the left (click the folder icon at the top of the panel if you cannot see it). Fill in **sections 1 to 5**. Come back and run the cell below when done.

In [ ]:
# Checks sections 1 to 5 are filled in before you continue
validate_recommendation(
    sections=[1, 2, 3, 4, 5],
    min_words={1: 6, 2: 10, 3: 10, 4: 12, 5: 12},
)

## Before you move to your decision

You have just described who bears the cost of each type of error and how those consequences shaped your thinking.

Before you open the file again, answer this for yourself, you do not need to write it down:

> Which error type did your reasoning in sections 1 to 5 lead you to treat as less acceptable? And does the model that reduces that error type happen to be the one you were already leaning towards, or did working through the consequences change your view?

*Click the triangle icon to the left to expand the section below.*

<details>
<summary>Quick reference: how the decision threshold works</summary>

The decision threshold is the probability at which the model switches from *no defect* to *defect*. You explored this above. Use what you found.

| Threshold | Flag if... | What changes |
|:--|:--|:--|
| 0.3 | 30% or more confident | More defects caught, more false alarms |
| 0.5 | 50% or more confident | Default setting |
| 0.7 | 70% or more confident | Fewer false alarms, more missed defects |

If you kept 0.5, explain why it is the right operating point for this domain, not just that it is the default. If you changed it, state what you found in the explorer and what it achieved.

</details>

Two of the remaining questions (oversight and accountability) sit outside pure ML. A few honest pointers:

**Section 8 (oversight):** 'Regular monitoring' is not sufficient. Name a specific mechanism, say who owns it, and say what it checks. Think about what would need to be true for someone to catch a problem with this model six months after deployment.

**Section 9 (accountability):** There is no single correct answer. Think about the difference between the person who built the model, the person who decided to deploy it, and the person who set the threshold. If you are unsure, searching *who is responsible for algorithmic deployment decisions* will show you this is a live debate, not a settled question.

Now complete **sections 6, 7, 8 and 9** in `recommendation.md`.

*This output maps to K26 and B4 in your occupational standard.*

In [ ]:
# Checks all nine sections
validate_recommendation(
    sections=[1, 2, 3, 4, 5, 6, 7, 8, 9],
    min_words={1: 6, 2: 10, 3: 10, 4: 12, 5: 12,
               6: 12, 7: 12, 8: 12, 9: 12},
)

## What you can do now

- Share `recommendation.md` with your coach or manager if you would like a second pair of eyes on your reasoning.
- Add a note to your learning journal that you have completed this lab and what your deployment decision was.
- If you want to revisit the reasoning behind your choice, everything is still here in the notebook.

## Extension

Is there a threshold, or a set of conditions, under which you would recommend that **neither model** be deployed at all?

If yes, describe the conditions precisely. If no, explain why not.

*This question is optional and is not validated.*